# HACA-Sent v3 — full run from scratch (Kaggle T4)

Retrains the in-domain encoder and evaluates **every model on all three test sets**:
`domaine_reel_v2` (human gold), `domaine_reel_v3` (rubric-aligned), `domaine_reel_v4_balanced`
(100/150/200 diagnostic). Use after a session restart (checkpoints are gone).

**Before running:** push the branch so the notebook can clone it —
`git push -u origin feat/haca-sent-v3`, then set `GITHUB_REPO` in cell 2.

Settings → Accelerator **GPU T4 x1**, Internet **On**. Core path ≈ 50 min; optionals add more.

> ⚠ On v4 the synthetic-positive F1 is style-memorisation (optimistic). The honest pos signal
> is the **v2** row. See report/HACA_TEST_V4_DATASHEET.md.

In [ ]:
# 1 — deps
import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable,'-m','pip','install','-q',*a])
pip('bitsandbytes>=0.43.0','pysrt>=1.1.2','seaborn>=0.13.0'); print('deps ok')

In [ ]:
# 2 — clone the branch (must be pushed to GitHub first)
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/haca-benchmark.git'  # <-- CHANGE ME
BRANCH      = 'feat/haca-sent-v3'
import os, sys, subprocess
WORK='/kaggle/working'; REPO=os.path.join(WORK,'benchmark')
if not os.path.isdir(REPO):
    subprocess.check_call(['git','clone','-b',BRANCH,GITHUB_REPO,REPO])
else:
    subprocess.check_call(['git','-C',REPO,'pull'])
os.chdir(REPO); sys.path.insert(0, os.path.join(REPO,'src'))
print('repo at', REPO)

In [ ]:
# 3 — MAC raw corpus (needed for marbertv2-haca = MAC + HACA-Sent)
import requests, os
os.makedirs('data/raw/MAC', exist_ok=True); dest='data/raw/MAC/MAC corpus.csv'
if not os.path.exists(dest):
    r=requests.get('https://raw.githubusercontent.com/LeMGarouani/MAC/master/MAC%20corpus.csv',timeout=60)
    r.raise_for_status(); open(dest,'wb').write(r.content); print('MAC', len(r.content)//1024,'KB')
else: print('MAC present')

In [ ]:
# 4 — GPU + verify all data is present
import torch, pandas as pd, subprocess, sys
from src.utils import set_seeds; set_seeds()
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
subprocess.check_call([sys.executable,'src/build_haca_train.py'])
for f in ['domaine_reel_v2','domaine_reel_v3','domaine_reel_v4_balanced']:
    d=pd.read_csv(f'data/test_sets/{f}.csv'); print(f'{f:28s} n={len(d):4d} dist={dict(d.label.value_counts())}')

## Train the in-domain encoder (from scratch)

In [ ]:
# 5 — train marbertv2-haca  (~30 min). Writes checkpoints/marbertv2-haca + v2/darija_ar results
import time
from src.finetune import finetune
t0=time.time(); finetune('marbertv2-haca'); print(f'{(time.time()-t0)/60:.1f} min')

In [ ]:
# 6 — calibrate thresholds on the gold v2
import subprocess, sys
subprocess.check_call([sys.executable,'src/calibrate_thresholds.py','--model','marbertv2-haca'])

## Evaluate the encoder on all three test sets
Reuses the harness; writes `results/marbertv2-haca_<testset>.json` for each.

In [ ]:
# 7 — encoder eval helper, run on v2 / v3 / v4 / darija_ar
from transformers import pipeline, AutoTokenizer
from label_maps import FINETUNED_MAP, apply_map
from harness import evaluate_model

def eval_encoder(model_key, ckpt, langs):
    tok=AutoTokenizer.from_pretrained(ckpt); tok.model_max_length=512
    pipe=pipeline('text-classification', model=ckpt, tokenizer=tok,
                  device=0 if __import__('torch').cuda.is_available() else -1, top_k=None)
    def predict_fn(texts):
        out=pipe(list(texts), batch_size=32, truncation=True)
        return [apply_map(max((x if isinstance(x,list) else [x]), key=lambda d:d['score'])['label'], FINETUNED_MAP) for x in out]
    for lang in langs:
        evaluate_model(model_key, lang, predict_fn)

eval_encoder('marbertv2-haca','checkpoints/marbertv2-haca',
             ['domaine_reel_v2','domaine_reel_v3','domaine_reel_v4_balanced','darija_ar'])

## Rubric-prompted LLM on all three test sets

In [ ]:
# 8 — Atlas-Chat-2B (rubric) on v2 / v3 / v4  (~10 min each, 4-bit)
import subprocess, sys
for t in ['domaine_reel_v2','domaine_reel_v3','domaine_reel_v4_balanced']:
    subprocess.check_call([sys.executable,'src/eval_llm_rubric.py','--model','2b','--test',f'data/test_sets/{t}.csv'])

## Optional — extra models

In [ ]:
# 9 — (optional) marbertv2-haca-only (in-domain only; continues from the checkpoint above)
from src.finetune import finetune
finetune('marbertv2-haca-only')
eval_encoder('marbertv2-haca-only','checkpoints/marbertv2-haca-only',
             ['domaine_reel_v2','domaine_reel_v3','domaine_reel_v4_balanced','darija_ar'])

In [ ]:
# 10 — (optional) darijabert-haca (permissive licence) + Atlas-Chat-9B
from src.finetune import finetune
finetune('darijabert-haca')
eval_encoder('darijabert-haca','checkpoints/darijabert-haca',
             ['domaine_reel_v2','domaine_reel_v3','domaine_reel_v4_balanced','darija_ar'])
import subprocess, sys
for t in ['domaine_reel_v2','domaine_reel_v3','domaine_reel_v4_balanced']:
    subprocess.check_call([sys.executable,'src/eval_llm_rubric.py','--model','9b','--test',f'data/test_sets/{t}.csv'])

## Consolidated results table

In [ ]:
# 11 — print macro-F1 + pos-F1 for every (model, test set)
import json, os
rows=[]
for f in sorted(os.listdir('results')):
    if not f.endswith('.json') or f.startswith('thresholds'): continue
    if not any(k in f for k in ['haca','rubric']): continue
    d=json.load(open('results/'+f))
    if 'macro_f1' not in d: continue
    rep=d.get('classification_report',{}); pos=rep.get('pos',{}).get('f1-score')
    model=d['model']; test=d['lang']
    rows.append((model,test,d['macro_f1'],pos,d.get('n')))
rows.sort()
print(f"{'model':22s} {'test':26s} {'macroF1':>8} {'posF1':>7} {'n':>5}")
print('-'*72)
for m,t,mf,pf,n in rows:
    print(f"{m:22s} {t:26s} {mf:8.3f} {('%.3f'%pf) if pf is not None else '   -':>7} {str(n):>5}")
print('\n(read v2 = honest gold; v4 pos-F1 is optimistic — synthetic style memorisation)')

In [ ]:
# 12 — (optional, tiny) zip results for download
import shutil; print(shutil.make_archive('/kaggle/working/results_all','zip','results'))